# 실험 D — LLM blind (이미지도 OCR도 없음)

**목적:** 질문과 선지만으로 몇 %를 맞히는지 측정합니다. 이것이 다른 모든 실험의 **진짜 바닥선**입니다.

무작위 추측은 25%지만, 선지에 언어적 단서가 있으면(예: `['조앤조','ESC','다이소','ON세라']`에서 가장 흔한 상호를 고르기)
그보다 훨씬 높게 나올 수 있습니다. D를 모르면 실험 A/B/C의 점수에서
**"모델이 실제로 읽어서 맞힌 몫"과 "언어 편향으로 찍어서 맞힌 몫"을 분리할 수 없습니다.**

- 답변 모델: 실험 A와 **동일한** Qwen3.5-4B (4bit NF4), 동일한 greedy 디코딩, 동일한 24토큰 제한
- 검증 분할: 실험 A가 저장한 `train_half_val.csv` 672문항을 그대로 사용 (ID 해시로 검증)
- 프롬프트/파서: `common_eval.py`에서 import — 코드 복사 없음
- A와 **단 하나** 다른 점: OCR 근거가 들어가지 않습니다

In [ ]:
# Resolve shared code and local assets from the repository root.
import os as _project_os
import sys as _project_sys
from pathlib import Path as _ProjectPath
_project_root = next((p for p in [_ProjectPath.cwd(), *_ProjectPath.cwd().parents]
                      if (p / 'src' / 'ocr_engine.py').is_file()), None)
if _project_root is not None:
    _project_os.chdir(_project_root)
    if str(_project_root / 'src') not in _project_sys.path:
        _project_sys.path.insert(0, str(_project_root / 'src'))
# End repository path setup.

import gc, json, time
from pathlib import Path

import torch
import pandas as pd
from tqdm.auto import tqdm

import common_eval as ce

ANSWER_MODEL_DIR = Path("downloads/models/Qwen3.5-4B")
OUTPUT_DIR = Path("output/exp_D_llm_blind")
MAX_NEW_TOKENS = 24      # 실험 A와 동일
ANSWER_4BIT = True       # 실험 A와 동일
SMOKE_TEST = False       # True면 앞 8문항만

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
import os
for key in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"):
    os.environ[key] = "1"

assert torch.cuda.is_available(), "이 실험 설정은 CUDA GPU를 사용합니다."
assert (ANSWER_MODEL_DIR / "config.json").is_file(), f"로컬 모델을 준비하세요: {ANSWER_MODEL_DIR}"
DEVICE = "cuda:0"
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
torch.manual_seed(42)
ce._self_test()
print("GPU:", torch.cuda.get_device_name(0), "dtype:", DTYPE)

## 검증 분할 로드
실험 A가 저장한 CSV를 그대로 읽습니다. 재계산하지 않으므로 sklearn 버전이 달라도 분할이 어긋날 수 없고,
`run_config.json`의 ID 해시와 대조해 **같은 672문항임을 증명**합니다.

In [ ]:
val_df = ce.load_val()
if SMOKE_TEST:
    print("⚠ SMOKE_TEST: 공식 비교 결과로 사용 금지")
    val_df = val_df.head(8).copy()
# 정답은 모델 입력에서 제거합니다.
rows = val_df.drop(columns=["answer"]).to_dict("records")
print("추론 대상:", len(rows))

## 프롬프트
질문/선지 렌더링은 실험 A와 **글자 단위로 동일**합니다(`ce.make_prompt`).
뒤에 붙는 한 문장만 다릅니다 — A는 OCR 근거를 제공한다고 알리고, D는 아무 근거도 없음을 알립니다.
이미지가 없는데 "이미지를 보고"라고만 두면 모델이 존재하지 않는 이미지를 가정하므로 명시합니다.

In [ ]:
def blind_messages(row):
    text = (ce.make_prompt(row) +
            "\n이 단계에는 이미지도 OCR 텍스트도 제공되지 않습니다. "
            "질문과 선지만 보고 가장 그럴듯한 하나를 고르세요. 반드시 한 글자로 답하세요.")
    return [{"role": "system", "content": [{"type": "text", "text": ce.SYSTEM}]},
            {"role": "user", "content": [{"type": "text", "text": text}]}]

print(blind_messages(rows[0])[1]["content"][0]["text"])

## 모델 로드
실험 A의 `load_answer()`와 동일한 설정입니다 (4bit NF4, double quant, compute dtype 동일).

In [ ]:
from transformers import AutoProcessor, Qwen3_5ForConditionalGeneration

def load_answer():
    proc = AutoProcessor.from_pretrained(str(ANSWER_MODEL_DIR), local_files_only=True)
    kwargs = dict(local_files_only=True, dtype=DTYPE, device_map={"": 0})
    if ANSWER_4BIT:
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
    model = Qwen3_5ForConditionalGeneration.from_pretrained(str(ANSWER_MODEL_DIR), **kwargs).eval()
    return model, proc

@torch.inference_mode()
def generate_text(model, proc, inputs, limit):
    inputs = inputs.to(model.device)
    prefix_len = inputs["input_ids"].shape[-1]
    output = model.generate(**inputs, max_new_tokens=limit, do_sample=False, use_cache=True)
    new = output[0, prefix_len:]
    eos = model.generation_config.eos_token_id
    eos_ids = [] if eos is None else ([eos] if isinstance(eos, int) else list(eos))
    truncated = len(new) >= limit and (not len(new) or int(new[-1]) not in eos_ids)
    return proc.decode(new, skip_special_tokens=True).strip(), bool(truncated)

## 추론
이미지 처리가 없어 빠릅니다. 결과는 한 문항씩 즉시 JSONL로 flush하므로 중간에 멈춰도 완료분은 남습니다.

In [ ]:
model, proc = load_answer()
torch.cuda.reset_peak_memory_stats()
results = []
try:
    with (OUTPUT_DIR / "predictions.jsonl").open("w", encoding="utf-8") as log:
        for row in tqdm(rows, desc="D: LLM blind"):
            torch.cuda.synchronize(); start = time.perf_counter()
            inputs = proc.apply_chat_template(
                blind_messages(row), add_generation_prompt=True, tokenize=True,
                return_dict=True, return_tensors="pt", enable_thinking=False)
            raw, capped = generate_text(model, proc, inputs, MAX_NEW_TOKENS)
            torch.cuda.synchronize(); elapsed = time.perf_counter() - start
            item = dict(id=row["id"], raw_output=raw, pred=ce.parse_choice(raw),
                        answer_truncated=capped, seconds=elapsed)
            results.append(item)
            log.write(json.dumps(item, ensure_ascii=False) + "\n"); log.flush()
            del inputs
    peak = torch.cuda.max_memory_allocated() / 1024**3
finally:
    del model, proc
    gc.collect(); torch.cuda.empty_cache()
print(f"완료 / GPU peak {peak:.2f} GiB")

## 분석
`common_eval.save_analysis`는 실험 A가 쓴 것과 같은 지표·같은 파일 형식을 냅니다.
INVALID는 임의로 채우지 않고 별도 라벨로 정확도 분모에 포함합니다.

In [ ]:
pred_df = val_df.merge(pd.DataFrame(results), on="id", validate="one_to_one", sort=False)
assert pred_df.id.tolist() == val_df.id.tolist()
pred_df["correct"] = pred_df.pred == pred_df.answer

score = ce.save_analysis(pred_df, "D_llm_blind", OUTPUT_DIR, extra=dict(
    answer_model=str(ANSWER_MODEL_DIR), answer_4bit=ANSWER_4BIT,
    answer_max_new_tokens=MAX_NEW_TOKENS, peak_allocated_gib=peak,
    answer_truncated_rate=float(pred_df.answer_truncated.mean()),
    uses_image=False, uses_ocr=False, smoke=SMOKE_TEST))
print()
print("무작위 추측 기준선: 0.2500")
print(f"D (언어 편향만):    {score['accuracy']:.4f}")
print(f"→ 선지만으로 얻는 이득: {score['accuracy'] - 0.25:+.4f}")
print()
print("이 값이 A/B/C 점수를 해석하는 기준입니다.")
print("예: A가 0.58이고 D가 0.42면 OCR의 순기여는 +0.16 입니다.")
print("산출물:", OUTPUT_DIR.resolve())